# 🏦 Notebook 03 — Feature Engineering

**Project:** Bank Loan Default Risk Analysis  
**Goal:** Encode categorical variables, create interaction features, scale numerical features, and produce a model-ready dataset.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cleaned/loan_data_cleaned.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

## 1. Encode Loan Grade (Ordinal)

Loan grade has a natural order (A=best → G=worst), so we use ordinal encoding.

In [ ]:
grade_order = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]
oe = OrdinalEncoder(categories=grade_order)
df['grade_encoded'] = oe.fit_transform(df[['grade']])

# Sub-grade ordinal encode
subgrades = sorted(df['sub_grade'].unique().tolist())
oe2 = OrdinalEncoder(categories=[subgrades])
df['sub_grade_encoded'] = oe2.fit_transform(df[['sub_grade']])

print('Grade encoding:')
print(df[['grade', 'grade_encoded']].drop_duplicates().sort_values('grade'))

## 2. Encode Employment Length (Ordinal)

In [ ]:
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10, 'Unknown': -1
}
df['emp_length_encoded'] = df['emp_length'].map(emp_map)
df['emp_length_encoded'] = df['emp_length_encoded'].fillna(-1)

print('Employment length encoding done.')
print(df[['emp_length', 'emp_length_encoded']].drop_duplicates().sort_values('emp_length_encoded'))

## 3. One-Hot Encode Nominal Categories

In [ ]:
# Home ownership — nominal
df = pd.get_dummies(df, columns=['home_ownership'], drop_first=True, prefix='home')

# Loan purpose — nominal
df = pd.get_dummies(df, columns=['purpose'], drop_first=True, prefix='purpose')

print(f'Shape after one-hot encoding: {df.shape}')
print('New dummy columns:', [c for c in df.columns if c.startswith('home_') or c.startswith('purpose_')])

## 4. Create Interaction & Risk Features

In [ ]:
# DTI × interest rate interaction (high DTI + high rate = compounding risk)
df['dti_x_int_rate'] = df['dti'] * df['int_rate']

# High revolving utilization flag
df['high_revol_util'] = (df['revol_util'] >= 75).astype(int)

# Thin credit file flag (< 3 years credit history)
df['thin_file'] = (df['credit_history_years'] < 3).astype(int)

# Income per month vs installment stress ratio
df['income_monthly'] = df['annual_inc'] / 12
df['payment_stress'] = df['installment'] / df['income_monthly']
df['payment_stress'] = df['payment_stress'].clip(0, 2)  # cap extreme values

# High DTI flag
df['high_dti'] = (df['dti'] > 43).astype(int)

print('Interaction features created:')
new_cols = ['dti_x_int_rate', 'high_revol_util', 'thin_file', 'payment_stress', 'high_dti']
print(df[new_cols].describe().round(3))

## 5. Drop Original Categorical Columns

In [ ]:
cols_to_drop = ['grade', 'sub_grade', 'emp_length', 'income_monthly']
df.drop(columns=cols_to_drop, inplace=True)

print(f'Shape after dropping original categoricals: {df.shape}')
print(f'Total features: {df.shape[1] - 1} (excl. target)')

## 6. Train-Test Split

In [ ]:
X = df.drop(columns=['default'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train):,} rows ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test size:  {len(X_test):,} rows ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nClass balance in train set:')
print(y_train.value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 7. Scale Numerical Features

In [ ]:
num_features = ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
                'revol_bal', 'revol_util', 'credit_history_years', 'loan_to_income',
                'dti_x_int_rate', 'payment_stress', 'open_acc', 'total_acc']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

print('Numerical features scaled using StandardScaler (fit on train only).')
print(f'\nFinal feature matrix shape: {X_train_scaled.shape}')

## 8. Save Feature-Engineered Data

In [ ]:
import pickle

# Save split datasets
X_train.to_csv('../data/cleaned/X_train.csv', index=False)
X_test.to_csv('../data/cleaned/X_test.csv', index=False)
y_train.to_csv('../data/cleaned/y_train.csv', index=False)
y_test.to_csv('../data/cleaned/y_test.csv', index=False)

# Save scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save full feature dataset
df.to_csv('../data/cleaned/loan_data_features.csv', index=False)

print('All files saved successfully.')
print(f'Feature columns ({len(X_train.columns)}):')
print(X_train.columns.tolist())

## 9. Summary

| Step | Result |
|------|--------|
| Grade encoding | Ordinal (A=0, G=6) |
| Employment encoding | Ordinal (< 1yr=0, 10+ yrs=10) |
| Nominal one-hot | home_ownership, purpose |
| New features | dti_x_int_rate, high_revol_util, thin_file, payment_stress, high_dti |
| Train / test split | 80% / 20%, stratified |
| Scaling | StandardScaler on 13 numerical features |
| Total features | 34 |

**Next step:** `04_model_training.ipynb`